<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a> 所著《<a href="https://mng.bz/lZ5B">从零开始构建推理模型</a>》一书的补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 附录D：使用更大的LLMs

本笔记本中正在使用的包：

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",  # for download functions
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.17
torch version: 2.10.0
tokenizers version: 0.21.4


- 主要章节使用 Qwen3 0.6B 基础模型，因为它是 Qwen3 系列中最小的模型，因此最容易在消费级硬件上运行
- 然而，附录 C 中相同的 `Qwen3Model` 实现也可以使用相同的从零开始的 PyTorch 模型代码来加载更大的密集型 Qwen3 检查点

&nbsp;
## D.1 更大规模的密集 Qwen3 配置

该仓库在 `reasoning_from_scratch.appendix_c`（[reasoning_from_scratch/appendix_c.py](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/appendix_c.py)）中包含了多个更大规模的稠密 Qwen3 模型（超越 0.6B 模型）的配置字典：

| 模型规模 | 配置字典 |
| --- | --- |
| 1.7B | `QWEN3_CONFIG_1_7B` |
| 4B | `QWEN3_CONFIG_4B` |
| 8B | `QWEN3_CONFIG_8B` |
| 14B | `QWEN3_CONFIG_14B` |
| 32B | `QWEN3_CONFIG_32B` |

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-d/Appendix_D_F01_raschka.webp" width="500px">

- 如上图所示，这些是"稠密"的 Qwen3 变体，可在单个 GPU 上运行
- 还有"稀疏"的混合专家（Mixture-of-Experts）Qwen3 变体，但本书代码不支持；不过，如果你对从零实现感兴趣，可以在此处找到：https://github.com/rasbt/LLMs-from-scratch/tree/main/ch05/11_qwen3
- 所有这些模型都采用与附录C中0.6B模型相同的总体架构模式
- 变化的是嵌入维度、层数、注意力头数以及前馈网络的隐藏层维度

- 作为粗略下限，使用 bfloat16 格式存储权重大约需要每个参数 2 字节
- 这意味着仅 checkpoint 权重的大小大约为：

| 模型规模 | bfloat16 格式下的粗略权重内存 |
| --- | --- |
| 1.7B | 约 3.4 GB |
| 4B | 约 8 GB |
| 8B | 约 16 GB |
| 14B | 约 28 GB |
| 32B | 约 64 GB |


- 实际运行时内存使用量更高，因为我们还需要内存用于激活值、临时缓冲区以及通常的KV缓存

&nbsp;
## D.2 下载更大的 checkpoints 概述

- 与主要章节中使用的0.6B检查点不同，官方发布的更大规模Qwen3模型通常以`safetensors`文件形式分发，有时会拆分为多个分片文件
- 用于加载这些文件的辅助函数`download_from_huggingface_from_snapshots`需要额外安装一些依赖包：

```bash
!uv add huggingface_hub safetensors
```

or

```bash
!pip install huggingface_hub safetensors
```

&nbsp;
## D.3 加载更大的基础模型

下载权重：

In [2]:
from pathlib import Path
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.appendix_c import (
    download_from_huggingface_from_snapshots
)


device = get_device()
local_dir = Path("qwen3-4b-base")

weights = download_from_huggingface_from_snapshots(
    repo_id="Qwen/Qwen3-4B-Base",
    local_dir=local_dir,
)

Using Apple Silicon GPU (MPS)


/Users/sebastian/Developer/reasoning-from-scratch/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 13 files: 100%|██████████████████████| 13/13 [00:00<00:00, 2616.79it/s]


- 初始化模型：

In [3]:
from reasoning_from_scratch.qwen3 import (
    Qwen3Model, load_hf_weights_into_qwen
)
from reasoning_from_scratch.appendix_c import QWEN3_CONFIG_4B


model = Qwen3Model(QWEN3_CONFIG_4B)
load_hf_weights_into_qwen(
    model,
    param_config={
        "n_layers": QWEN3_CONFIG_4B["n_layers"],
        "hidden_dim": QWEN3_CONFIG_4B["hidden_dim"],
    },
    params=weights,
)
model.to(device)
model.eval()

Model uses weight tying.


Qwen3Model(
  (tok_emb): Embedding(151936, 2560)
  (trf_blocks): ModuleList(
    (0-35): 36 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=2560, out_features=4096, bias=False)
        (W_key): Linear(in_features=2560, out_features=1024, bias=False)
        (W_value): Linear(in_features=2560, out_features=1024, bias=False)
        (out_proj): Linear(in_features=4096, out_features=2560, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=2560, out_features=9728, bias=False)
        (fc2): Linear(in_features=2560, out_features=9728, bias=False)
        (fc3): Linear(in_features=9728, out_features=2560, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=2560, out_features=151936, bias=False)
)

- 加载 tokenizer：

In [4]:
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer
import shutil

# Note that the original base tokenizer is called "tokenizer.json"
# We rename it to distinguish from the reasoning tokenizer (next section)
tokenizer_src = local_dir / "tokenizer.json"
tokenizer_path = local_dir / "tokenizer-base.json"

if not tokenizer_path.exists():
    shutil.copyfile(tokenizer_src, tokenizer_path)

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

- 使用模型：

In [5]:
import torch
from reasoning_from_scratch.ch02 import (
    generate_text_basic_stream_cache,
)

prompt = "Explain large language models in two sentences."
input_ids = torch.tensor(
    tokenizer.encode(prompt),
    device=device,
).unsqueeze(0)

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=64,
    eos_token_id=tokenizer.eos_token_id,
):
    print(tokenizer.decode(token.squeeze(0).tolist()), end="", flush=True)

 Large language models are artificial intelligence systems that use deep learning techniques to understand and generate human-like text. They are trained on vast amounts of data and can perform a wide range of natural language processing tasks, such as translation, summarization, and question answering.

&nbsp;
## D.4 加载更大的推理变体

- 同样的思路也适用于更大规模的推理风格 Qwen3 模型
- 给定模型大小的架构保持不变；仅检查点和分词器设置会改变

例如，要加载4B推理变体而非4B基础变体，我们需要：

- 将仓库ID从 `Qwen/Qwen3-4B-Base` 切换为 `Qwen/Qwen3-4B`；
- 将 `tokenizer.json` 文件复制为 `tokenizer-reasoning.json`；
- 按如下方式初始化分词器：

```python
tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=tokenizer_path,
    apply_chat_template=True,
    add_generation_prompt=True,
    add_thinking=True,
)
```

- 模型加载和使用的其余代码保持不变

&nbsp;
## D.5 实践建议

- 本节无代码